# 🧠 Module 4: Face-to-Body Association Pipeline

**Role in the project:** Core video-processing engine of the Person Re-ID system. It detects and tracks every person in a camera feed, extracts body (CLIP-ReID) and face (ArcFace) embeddings, resolves a persistent global identity per person, and retroactively merges wrong body-based IDs whenever a face is recognized.

**Architecture:** RT-DETR (detection) + BotSort (tracking) + YOLOv11-Pose (skeleton quality gate) + CLIP-ReID (1280-D body) + InsightFace ArcFace (512-D face) + Self-Adaptive Gallery + Dual Pinecone indices

**Pipeline:** Face/Body Spatial Merge → Face Ground Truth Query → Retroactive Identity Healing → True Drop-Cost Gallery Optimization

**Inputs:** a supermarket camera video (`.mp4`) and, optionally, a previously saved `identity_buffer.pkl`

**Outputs:**
- `identity_buffer.pkl` — body gallery consumed by **Module 5**
- Pinecone `supermarket-reid` (body) and `supermarket-faces` (face) indices
- MongoDB `supermarket_db.persons` documents (sightings + faces)
- ImageKit CDN URLs for the best body/face crops

**Depends on:** Module 2/3 memory format (`identity_buffer*.pkl`)
**Feeds into:** Module 5 (search engine) and Module 6 (dashboard)


## 1. Environment Setup

In [ ]:
%%capture
!pip install ultralytics boxmot opencv-python scipy
!pip install pinecone imagekitio scikit-learn pymongo
!pip install insightface onnxruntime-gpu

print("Dependencies Ready.")

## 2. Global Configuration & Parameters

In [ ]:
import os, shutil, cv2, time, torch, pickle
import numpy as np
import concurrent.futures
from pathlib import Path
from datetime import datetime, timezone
from tqdm.auto import tqdm
import torch.nn.functional as F
from sklearn.cluster import DBSCAN
from pymongo import MongoClient
from kaggle_secrets import UserSecretsClient

# ═══════════════════════════════════════════════════════════════
# CAMERA MODE — set False for non-face cameras
# ═══════════════════════════════════════════════════════════════
IS_FACE_CAMERA = True  # enables face detection, face-pinecone queries, and identity merge logic

# ═══════════════════════════════════════════════════════════════
# PATHS & VIDEO
# ═══════════════════════════════════════════════════════════════
INPUT_VIDEO = "/kaggle/input/datasets/abderrahimzineddine/reid-testing-samples/store_cam1_mirrored.mp4"
OUTPUT_DIR = Path("/kaggle/working/module2_database")
DB_IMAGES_DIR = OUTPUT_DIR / "saved_identities"
CAMERA_ID = "Face_Cam_Entrance"  # human-readable name for this camera

# Simulated start time of the video (anchors frame→timestamp conversion)
VIDEO_START_TIME = datetime(2026, 5, 5, 9, 0, 0, tzinfo=timezone.utc)

# ═══════════════════════════════════════════════════════════════
# CREDENTIALS
# ═══════════════════════════════════════════════════════════════
try:
    secrets = UserSecretsClient()
    IMAGEKIT_URL_ENDPOINT = secrets.get_secret("IMAGEKIT_URL_ENDPOINT")
    IMAGEKIT_PUBLIC_KEY   = secrets.get_secret("IMAGEKIT_PUBLIC_KEY")
    IMAGEKIT_PRIVATE_KEY  = secrets.get_secret("IMAGEKIT_PRIVATE_KEY")
    MONGO_URI             = secrets.get_secret("MONGO_URI")
    PINECONE_API_KEY      = secrets.get_secret("PINECONE_API_KEY")
except Exception:
    IMAGEKIT_PRIVATE_KEY = "YOUR_IMAGEKIT_PRIVATE"
    MONGO_URI            = "YOUR_MONGO_URI"
    PINECONE_API_KEY     = "YOUR_PINECONE_KEY"

# ═══════════════════════════════════════════════════════════════
# PINECONE INDICES
# ═══════════════════════════════════════════════════════════════
PINECONE_INDEX_NAME = "supermarket-reid"           # 1280D cosine – body ReID embeddings
PINECONE_FACE_INDEX_NAME = "supermarket-faces"     # 512D cosine – face embeddings (must be manually created in Pinecone console)

# ═══════════════════════════════════════════════════════════════
# SESSION CONTROL
# ═══════════════════════════════════════════════════════════════
RESET_DATABASE = False  # True = wipe MongoDB + local pickle + saved crops

# ═══════════════════════════════════════════════════════════════
# ReID THRESHOLDS
# ═══════════════════════════════════════════════════════════════
REID_CONF_THRESHOLD = 0.75   # minimum detector confidence to consider a body crop
POSE_CONF_THRESHOLD = 0.3    # per-keypoint visibility threshold
MIN_VISIBLE_RATIO   = 0.5    # fraction of keypoints that must be visible (quality gate)

# ═══════════════════════════════════════════════════════════════
# FACE THRESHOLDS (only used when IS_FACE_CAMERA = True)
# ═══════════════════════════════════════════════════════════════
FACE_CONF_THRESHOLD  = 0.7   # InsightFace det_score minimum (raised from 0.5 for reliability)
FACE_MATCH_THRESHOLD = 0.55  # Pinecone cosine similarity to accept a face match

# ═══════════════════════════════════════════════════════════════
# GALLERY & UDM
# ═══════════════════════════════════════════════════════════════
TAU          = 2.0   # softmax ratio threshold for confident live classification
MIN_COS_SIM  = 0.78  # minimum cosine similarity to even consider a gallery match
MAX_IMAGES   = 50    # max crops per identity in the self-adaptive gallery
UDM_MIN_SIZE = 5     # minimum cluster size for DBSCAN in the Unknown Data Manager
UDM_EPSILON  = 0.15  # DBSCAN distance epsilon (1 - cosine_similarity)

# ── PKL buffer auto-detection ──
input_pkls = list(Path('/kaggle/input').rglob('identity_buffer*.pkl'))
if input_pkls and not RESET_DATABASE:
    LOAD_BUFFER_PATH = input_pkls[0]
    print(f"Found imported memory buffer: {LOAD_BUFFER_PATH}")
else:
    LOAD_BUFFER_PATH = Path('/kaggle/working/identity_buffer.pkl')

SAVE_BUFFER_PATH = Path('/kaggle/working/identity_buffer.pkl')

if RESET_DATABASE:
    if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
    if SAVE_BUFFER_PATH.exists(): SAVE_BUFFER_PATH.unlink()

DB_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
print("Configuration Loaded. Folders Ready.")

## 3. Model Initialization & Cloud Connections

In [ ]:
from ultralytics import RTDETR, YOLO
from boxmot.trackers.botsort.botsort import BotSort
from imagekitio import ImageKit
from pinecone import Pinecone
import boxmot, pkgutil, inspect

if IS_FACE_CAMERA:
    from insightface.app import FaceAnalysis

n_gpus = torch.cuda.device_count()
DET_DEVICE  = 'cuda:0'
POSE_DEVICE = 'cuda:1' if n_gpus > 1 else 'cuda:0'
REID_DEVICE = 'cuda:1' if n_gpus > 1 else 'cuda:0'

print("Loading Models...")
detector = RTDETR('rtdetr-l.pt').to(DET_DEVICE)
tracker  = BotSort(reid_weights='osnet_x0_25_msmt17.pt', device=DET_DEVICE, half=False)
pose_model = YOLO('yolo26s-pose.pt').to(POSE_DEVICE)

# Dynamically find the ReID class from boxmot
ReID = None
for importer, modname, ispkg in pkgutil.walk_packages(boxmot.__path__, boxmot.__name__ + '.'):
    try:
        mod = __import__(modname, fromlist="dummy")
        for name, obj in inspect.getmembers(mod):
            if inspect.isclass(obj) and name in ['ReID', 'ReIDDetectMultiBackend']:
                ReID = obj; break
    except Exception: continue
    if ReID is not None: break

reid_model = ReID(weights='clip_market1501.pt', device=REID_DEVICE, half=False)

# Face engine (only for face cameras)
face_app = None
face_index = None
if IS_FACE_CAMERA:
    print("Loading Face Engine (InsightFace/ArcFace)...")
    face_app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    face_app.prepare(ctx_id=0, det_size=(640, 640))

print("Connecting to Databases...")
imagekit = ImageKit(private_key=IMAGEKIT_PRIVATE_KEY)
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)

if IS_FACE_CAMERA:
    face_index = pc.Index(PINECONE_FACE_INDEX_NAME)

mongo_client = MongoClient(MONGO_URI)
db = mongo_client["supermarket_db"]
persons_col = db["persons"]

if RESET_DATABASE:
    persons_col.delete_many({})
    print("MongoDB cleared.")

## 4. Database Architecture Classes

In [ ]:
class SightingTracker:
    """Tracks per-track sighting metadata: timestamps, best crops, face data."""
    def __init__(self, fps=30):
        self.fps = fps
        self.active_sightings = {}
        self.completed_sightings = []
        self.active_faces = {}
        self.completed_faces = []

    def _frame_to_ts(self, frame):
        """Convert frame number to UTC timestamp anchored to VIDEO_START_TIME."""
        offset_sec = frame / self.fps
        return VIDEO_START_TIME.timestamp() + offset_sec

    def update_track(self, tid, frame, conf, ratio, img, gid=None, bbox=None):
        if tid not in self.active_sightings:
            self.active_sightings[tid] = {
                "camera_id": CAMERA_ID,
                "track_id": tid,
                "global_id": gid,
                "entry_ts": self._frame_to_ts(frame),
                "exit_ts": self._frame_to_ts(frame),
                "duration_sec": 0.0,
                "frame_start": frame,
                "frame_end": frame,
                "n_detections": 1,
                "sum_det_score": float(conf),
                "sum_skeleton_ratio": float(ratio),
                "best_crop_quality": float(conf * ratio),
                "best_crop_img": img,
                "best_crop_bbox": list(bbox) if bbox else None,
                "best_crop_frame": frame
            }
        else:
            s = self.active_sightings[tid]
            if gid is not None and s["global_id"] is None:
                s["global_id"] = gid
            s["exit_ts"] = self._frame_to_ts(frame)
            s["frame_end"] = frame
            s["duration_sec"] = s["exit_ts"] - s["entry_ts"]
            s["n_detections"] += 1
            s["sum_det_score"] += float(conf)
            s["sum_skeleton_ratio"] += float(ratio)
            q = float(conf * ratio)
            if q > s["best_crop_quality"]:
                s["best_crop_quality"] = q
                s["best_crop_img"] = img
                s["best_crop_bbox"] = list(bbox) if bbox else None
                s["best_crop_frame"] = frame

    def update_face(self, tid, face_img, face_emb, face_conf):
        """Track face data separately from body sightings."""
        fc = float(face_conf)
        if tid not in self.active_faces:
            self.active_faces[tid] = {
                "best_face_img": face_img,
                "best_face_emb": face_emb,
                "best_face_conf": fc
            }
        elif fc > self.active_faces[tid]["best_face_conf"]:
            self.active_faces[tid]["best_face_img"] = face_img
            self.active_faces[tid]["best_face_emb"] = face_emb
            self.active_faces[tid]["best_face_conf"] = fc

    def assign_global_id(self, tid, gid):
        if tid in self.active_sightings:
            self.active_sightings[tid]["global_id"] = gid

    def close_track(self, tid):
        if tid in self.active_sightings:
            s = self.active_sightings.pop(tid)
            face = self.active_faces.pop(tid, None)
            if s["global_id"] is not None:
                s["avg_det_score"] = float(s["sum_det_score"] / s["n_detections"])
                s["avg_skeleton_ratio"] = float(s["sum_skeleton_ratio"] / s["n_detections"])
                del s["sum_det_score"]
                del s["sum_skeleton_ratio"]
                self.completed_sightings.append(s)
                if face is not None:
                    face["global_id"] = s["global_id"]
                    face["track_id"] = tid
                    face["camera_id"] = s["camera_id"]
                    face["timestamp"] = s["exit_ts"]
                    self.completed_faces.append(face)


class UnknownDataManager:
    """Pools unresolved body crops and clusters them via DBSCAN."""
    def __init__(self, eps, min_samples):
        self.pool = []
        self.eps = eps
        self.min_samples = min_samples

    def add(self, item):
        self.pool.append(item)

    def run_dbscan(self, gallery):
        if len(self.pool) < self.min_samples:
            return []

        embs = torch.stack([x['emb'] for x in self.pool])
        sim_matrix = torch.mm(embs, embs.t()).cpu().numpy()
        dist_matrix = np.clip(1.0 - sim_matrix, 0.0, 2.0)

        clustering = DBSCAN(eps=self.eps, min_samples=self.min_samples, metric='precomputed').fit(dist_matrix)
        labels = clustering.labels_

        actions = []
        new_pool = []

        for lbl in set(labels):
            if lbl == -1: continue
            indices = np.where(labels == lbl)[0]
            cluster_items = [self.pool[i] for i in indices]
            cluster_track_ids = list(set([x['track_id'] for x in cluster_items]))

            # Face VIP bypass: if any item in this cluster has a forced_gid, honor it
            forced_gids = [x.get('forced_gid') for x in cluster_items if x.get('forced_gid') is not None]
            if forced_gids:
                best_gid = max(set(forced_gids), key=forced_gids.count)
                # Ensure the gallery class exists (could be from a previous session via Pinecone)
                if best_gid not in gallery.classes:
                    gallery.mint_class_at(best_gid)
                for item in cluster_items:
                    gallery.add_observation(best_gid, item)
                actions.append(("FACE_VIP_BYPASS", best_gid, len(cluster_items), cluster_track_ids))
                continue

            cluster_mean = F.normalize(torch.mean(torch.stack([x['emb'] for x in cluster_items]), dim=0), p=2, dim=0)

            merged = False
            if gallery.num_classes() > 0:
                centroids, active_gids = gallery.get_weighted_centroids()
                if centroids is not None:
                    sims = torch.mv(centroids, cluster_mean)
                    best_sim, best_idx = torch.max(sims, dim=0)

                    if (1.0 - best_sim.item()) <= self.eps:
                        best_gid = active_gids[best_idx.item()]
                        for item in cluster_items:
                            gallery.add_observation(best_gid, item)
                        actions.append(("MERGE", best_gid, len(cluster_items), cluster_track_ids))
                        merged = True

            if not merged:
                new_gid = gallery.mint_new_class()
                for item in cluster_items:
                    gallery.add_observation(new_gid, item)
                actions.append(("MINT", new_gid, len(cluster_items), cluster_track_ids))

        for i, lbl in enumerate(labels):
            if lbl == -1: new_pool.append(self.pool[i])
        self.pool = new_pool
        return actions


class GalleryManager:
    """Self-adaptive gallery with drop-cost budget optimization."""
    def __init__(self, max_images):
        self.classes = {}
        self.gids = []
        self.max_images = max_images
        self.current_max_id = 0

    def num_classes(self):
        return len(self.gids)

    def mint_new_class(self):
        self.current_max_id += 1
        gid = self.current_max_id
        self._init_class(gid)
        return gid

    def mint_class_at(self, gid):
        """Create a gallery slot at a specific gid (used by forced_gid / face bypass)."""
        if gid in self.classes:
            return
        self.classes[gid] = {'embs': [], 'ratios': [], 'filepaths': [], 'scores': []}
        self.gids.append(gid)
        self.current_max_id = max(self.current_max_id, gid)
        (DB_IMAGES_DIR / f"global_id_{gid:03d}").mkdir(exist_ok=True)

    def _init_class(self, gid):
        self.classes[gid] = {'embs': [], 'ratios': [], 'filepaths': [], 'scores': []}
        self.gids.append(gid)
        (DB_IMAGES_DIR / f"global_id_{gid:03d}").mkdir(exist_ok=True)

    def get_weighted_centroids(self):
        """Returns (centroids_tensor, active_gids_list) skipping empty classes."""
        centroids = []
        active_gids = []
        for gid in self.gids:
            data = self.classes[gid]
            if not data['embs']:
                continue
            embs = torch.stack(data['embs'])
            ratios = torch.tensor(data['ratios'], device=embs.device).unsqueeze(1)
            w_sum = torch.sum(embs * ratios, dim=0)
            centroids.append(F.normalize(w_sum, p=2, dim=0))
            active_gids.append(gid)
        if not centroids:
            return None, []
        return torch.stack(centroids), active_gids

    def classify_live_track(self, emb, tau, exclude_gids=None):
        if self.num_classes() == 0: return None, 0.0, 0.0

        centroids, active_gids = self.get_weighted_centroids()
        if centroids is None: return None, 0.0, 0.0
        sims = torch.mv(centroids, emb)

        if exclude_gids:
            for i, gid in enumerate(active_gids):
                if gid in exclude_gids: sims[i] = -float('inf')

        probs = F.softmax(sims * 15.0, dim=0)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)

        max_prob = sorted_probs[0].item()
        best_sim = sims[sorted_idx[0]].item()
        best_gid = active_gids[sorted_idx[0].item()]

        if best_sim == -float('inf'): return None, 0.0, 0.0
        if best_sim < MIN_COS_SIM: return best_gid, 0.0, max_prob

        if len(sorted_probs) > 1: ratio = max_prob / (sorted_probs[1].item() + 1e-9)
        else: ratio = float('inf')

        return best_gid, ratio, max_prob

    def add_observation(self, gid, item):
        data = self.classes[gid]
        data['embs'].append(item['emb'])
        data['ratios'].append(item['ratio'])
        data['scores'].append(item['score'])

        filename = f"frame_{item['frame']:06d}_trk_{item['track_id']:03d}.jpg"
        filepath = DB_IMAGES_DIR / f"global_id_{gid:03d}" / filename
        cv2.imwrite(str(filepath), item['img'])
        data['filepaths'].append(filepath)
        self._optimize_budget(gid)

    def _optimize_budget(self, gid):
        """True self-adaptive gallery drop cost: redundancy + cross-gallery entropy."""
        data = self.classes[gid]
        if len(data['embs']) <= self.max_images: return

        embs = torch.stack(data['embs'])

        # Redundancy: max similarity to any OTHER crop of the same identity
        sim_matrix = torch.mm(embs, embs.t())
        sim_matrix.fill_diagonal_(-1.0)
        redundancy, _ = torch.max(sim_matrix, dim=1)

        # Uncertainty: entropy over softmax similarities to ALL gallery centroids
        centroids, _ = self.get_weighted_centroids()
        if centroids is not None and centroids.shape[0] > 1:
            sims_to_all = torch.mm(embs, centroids.t())
            probs = F.softmax(sims_to_all * 15.0, dim=1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=1)
            # FIX: normalize by log(N_classes), not log(embedding_dim)
            uncertainty = entropy / torch.log(torch.tensor(float(centroids.shape[0]), device=embs.device))
        else:
            uncertainty = torch.zeros(embs.shape[0], device=embs.device)

        # Drop the crop with highest combined cost (most redundant + most uncertain)
        cost = 0.5 * uncertainty.cpu().numpy() + 0.5 * redundancy.cpu().numpy()
        worst_idx = int(np.argmax(cost))

        # Safety: ensure all arrays are the same length before popping
        min_len = min(len(data[k]) for k in ['embs', 'ratios', 'scores', 'filepaths'])
        if worst_idx >= min_len:
            worst_idx = min_len - 1

        worst_filepath = Path(data['filepaths'][worst_idx])
        if worst_filepath.exists() and 'working' in str(worst_filepath):
            worst_filepath.unlink()

        for key in ['embs', 'ratios', 'scores', 'filepaths']:
            if worst_idx < len(data[key]):
                data[key].pop(worst_idx)


# ── Instantiate core objects ──
udm = UnknownDataManager(eps=UDM_EPSILON, min_samples=UDM_MIN_SIZE)
gallery = GalleryManager(max_images=MAX_IMAGES)
sighting_tracker = SightingTracker(fps=30)

def create_person_document(gid, first_seen_ts, col):
    if not col.find_one({"global_id": gid}):
        col.insert_one({
            "global_id": gid,
            "first_seen_ts": float(first_seen_ts),
            "last_seen_ts": float(first_seen_ts),
            "faces": [],
            "sightings": []
        })

## 5. Dual Face & Body Processing Loop

In [ ]:
def extract_reid_features(crops):
    with torch.no_grad():
        embeddings = reid_model(crops)
        if isinstance(embeddings, np.ndarray): embeddings = torch.tensor(embeddings)
        embeddings = F.normalize(embeddings, p=2, dim=1).to(REID_DEVICE)
    return embeddings

def get_spatially_matched_face(body_box, faces):
    """Returns the best face whose center falls in the top 40% of the body box."""
    x1, y1, x2, y2 = body_box
    body_top_y2 = y1 + (y2 - y1) * 0.4
    best_face, best_score = None, 0
    for f in faces:
        fx1, fy1, fx2, fy2 = f.bbox
        fcx, fcy = (fx1 + fx2) / 2, (fy1 + fy2) / 2
        if x1 <= fcx <= x2 and y1 <= fcy <= body_top_y2:
            if f.det_score > best_score:
                best_score = f.det_score
                best_face = f
    return best_face

def search_face_pinecone(face_emb_numpy):
    """Query the supermarket-faces index for a known identity."""
    res = face_index.query(vector=face_emb_numpy.tolist(), top_k=1, include_metadata=True)
    if res.matches and res.matches[0].score > FACE_MATCH_THRESHOLD:
        return int(res.matches[0].metadata['global_id'])
    return None

def execute_identity_merge(target_gid, old_gid):
    """Retroactive merge: reassigns tracks, gallery data, and MongoDB docs from old_gid → target_gid."""
    tqdm.write(f"🧬 RETROACTIVE HEAL: Merging ID {old_gid} → True Face ID {target_gid}")

    # Remap all local tracks
    for tid, gid in list(local_to_global_map.items()):
        if gid == old_gid:
            local_to_global_map[tid] = target_gid
            sighting_tracker.assign_global_id(tid, target_gid)

    # Merge gallery arrays
    if old_gid in gallery.classes:
        # Guard: ensure target exists in gallery (could be stale Pinecone hit)
        if target_gid not in gallery.classes:
            gallery.mint_class_at(target_gid)

        o_data = gallery.classes.pop(old_gid)
        gallery.gids.remove(old_gid)

        t_data = gallery.classes[target_gid]
        target_dir = DB_IMAGES_DIR / f"global_id_{target_gid:03d}"

        # Keep all arrays in sync: only merge entries whose file exists
        for j in range(len(o_data['embs'])):
            fp = Path(o_data['filepaths'][j]) if j < len(o_data['filepaths']) else None
            if fp and fp.exists():
                new_fp = target_dir / fp.name
                if 'input' in str(fp):
                    shutil.copy(str(fp), str(new_fp))
                else:
                    shutil.move(str(fp), str(new_fp))
                t_data['embs'].append(o_data['embs'][j])
                t_data['ratios'].append(o_data['ratios'][j])
                t_data['scores'].append(o_data['scores'][j])
                t_data['filepaths'].append(new_fp)

        # Prune back to budget
        while len(t_data['embs']) > gallery.max_images:
            gallery._optimize_budget(target_gid)

    # Merge MongoDB documents (ImageKit files stay in place — flat folder, URLs don't change)
    old_doc = persons_col.find_one({"global_id": old_gid})
    if old_doc:
        persons_col.update_one(
            {"global_id": target_gid},
            {"$push": {
                "sightings": {"$each": old_doc.get("sightings", [])},
                "faces": {"$each": old_doc.get("faces", [])}
            }}
        )
        persons_col.delete_one({"global_id": old_gid})
        
        # Update Pinecone face vectors: reassign old_gid → target_gid
        try:
            dummy_vec = [0.0] * 512
            res = face_index.query(vector=dummy_vec, top_k=100, filter={"global_id": old_gid}, include_metadata=True)
            if res.matches:
                for match in res.matches:
                    new_meta = dict(match.metadata)
                    new_meta["global_id"] = target_gid
                    face_index.update(id=match.id, set_metadata=new_meta)
                tqdm.write(f"    [Pinecone] Updated {len(res.matches)} face vectors → GID {target_gid}")
        except Exception as e:
            tqdm.write(f"    [Pinecone] Failed to update face metadata: {e}")


# ── Hydrate gallery from previous session ──
if LOAD_BUFFER_PATH.exists() and not RESET_DATABASE:
    print(f"\n💧 Hydrating RAM from previous session ({LOAD_BUFFER_PATH})...")
    with open(LOAD_BUFFER_PATH, 'rb') as f: saved_data = pickle.load(f)

    input_img_dirs = list(Path('/kaggle/input').rglob('saved_identities*'))
    input_img_base = input_img_dirs[0] if input_img_dirs else None

    max_id = 0
    for gid, data in saved_data.items():
        gallery.gids.append(gid)
        max_id = max(max_id, gid)

        # Keep all arrays in sync: only load entries whose filepath resolves
        valid_embs, valid_ratios, valid_scores, valid_paths = [], [], [], []
        for j, p in enumerate(data["filepaths"]):
            old_p = Path(p)
            resolved = None
            if old_p.exists():
                resolved = old_p
            elif input_img_base:
                new_p = input_img_base / old_p.parent.name / old_p.name
                if new_p.exists():
                    resolved = new_p
            if resolved:
                valid_paths.append(resolved)
                valid_embs.append(torch.tensor(data["embeddings"][j]).to(REID_DEVICE))
                valid_ratios.append(data["ratios"][j])
                valid_scores.append(data["det_scores"][j])

        # Skip identities with no resolvable files (avoids empty gallery slots)
        if not valid_embs:
            gallery.gids.remove(gid)
            continue

        gallery.classes[gid] = {
            'embs': valid_embs,
            'ratios': valid_ratios,
            'scores': valid_scores,
            'filepaths': valid_paths
        }
    gallery.current_max_id = max_id
    print(f"✅ Hydrated {gallery.num_classes()} body identities.")


# ═══════════════════════════════════════════════════════════════
# MAIN PROCESSING LOOP
# ═══════════════════════════════════════════════════════════════
cap = cv2.VideoCapture(INPUT_VIDEO)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
local_to_global_map = {}

with tqdm(total=TOTAL_FRAMES, desc="Multi-Modal Tracker", unit="fr") as pbar:
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1

        # 1. Body detection
        results = detector(frame, classes=[0], verbose=False)
        boxes = results[0].boxes
        if boxes is None or len(boxes) == 0:
            pbar.update(1); continue

        # Face detection (only on face cameras)
        faces_in_frame = face_app.get(frame) if IS_FACE_CAMERA else []

        dets = torch.cat([boxes.xyxy, boxes.conf.unsqueeze(1), boxes.cls.unsqueeze(1)], dim=1).cpu().numpy()
        tracks = tracker.update(dets, frame)
        if tracks is None or len(tracks) == 0:
            pbar.update(1); continue

        # 2. Filter by confidence
        valid_data = []
        for t in tracks:
            track_id = int(t[4])
            conf = float(t[5]) if len(t) > 5 else 1.0
            if conf >= REID_CONF_THRESHOLD:
                x1, y1, x2, y2 = map(int, t[:4])
                x1, y1 = max(0, x1), max(0, y1)
                crop = frame[y1:y2, x1:x2]
                if crop.size > 0:
                    valid_data.append({'track_id': track_id, 'conf': conf, 'box': [x1, y1, x2, y2], 'crop': crop})

        if valid_data:
            # 3. Pose quality gate
            pose_results = pose_model([d['crop'] for d in valid_data], verbose=False)
            pristine_data = []
            for i, res in enumerate(pose_results):
                visible_count = sum(1 for x, y, c in res.keypoints.data[0].cpu().numpy() if c > POSE_CONF_THRESHOLD) if res.keypoints is not None and len(res.keypoints.data) > 0 else 0
                ratio = visible_count / 17.0
                if ratio >= MIN_VISIBLE_RATIO:
                    pristine_data.append({**valid_data[i], 'ratio': ratio, 'score': valid_data[i]['conf']})

            if pristine_data:
                # 4. Body embeddings
                embeddings = extract_reid_features([d['crop'] for d in pristine_data])
                claimed_gids_this_frame = set()
                unresolved_items = []

                # 5. Identity resolution
                for i, d in enumerate(pristine_data):
                    item = {'emb': embeddings[i], 'ratio': d['ratio'], 'score': d['score'],
                            'img': d['crop'], 'track_id': d['track_id'], 'frame': frame_count}
                    tid = d['track_id']

                    gid = local_to_global_map.get(tid)
                    sighting_tracker.update_track(tid, frame_count, item['score'], item['ratio'], item['img'], gid, d['box'])

                    # ── Face logic (only on face cameras) ──
                    face_truth_gid = None
                    if IS_FACE_CAMERA:
                        matched_face = get_spatially_matched_face(d['box'], faces_in_frame)
                        if matched_face and matched_face.det_score > FACE_CONF_THRESHOLD:
                            fx1, fy1, fx2, fy2 = map(int, matched_face.bbox)
                            face_img = frame[max(0, fy1):fy2, max(0, fx1):fx2]
                            sighting_tracker.update_face(tid, face_img, matched_face.normed_embedding, matched_face.det_score)
                            face_truth_gid = search_face_pinecone(matched_face.normed_embedding)

                            if face_truth_gid:
                                if gid and gid != face_truth_gid:
                                    execute_identity_merge(face_truth_gid, gid)
                                    gid = face_truth_gid
                                elif not gid:
                                    item['forced_gid'] = face_truth_gid

                    # Route to gallery or UDM
                    gid = local_to_global_map.get(tid)
                    if gid is not None:
                        if gid not in claimed_gids_this_frame:
                            claimed_gids_this_frame.add(gid)
                            gallery.add_observation(gid, item)
                        else:
                            del local_to_global_map[tid]
                            unresolved_items.append(item)
                    else:
                        unresolved_items.append(item)

                # 6. Handle unknowns
                for item in unresolved_items:
                    tid = item['track_id']

                    # FIX: check forced_gid BEFORE the empty-gallery fallback
                    forced = item.get('forced_gid')
                    if forced is not None:
                        # Face recognized this person — honor the face ID regardless of gallery state
                        if forced not in gallery.classes:
                            gallery.mint_class_at(forced)
                        local_to_global_map[tid] = forced
                        claimed_gids_this_frame.add(forced)
                        gallery.add_observation(forced, item)
                        sighting_tracker.assign_global_id(tid, forced)
                        create_person_document(forced, sighting_tracker._frame_to_ts(frame_count), persons_col)
                    elif gallery.num_classes() == 0:
                        new_gid = gallery.mint_new_class()
                        local_to_global_map[tid] = new_gid
                        claimed_gids_this_frame.add(new_gid)
                        gallery.add_observation(new_gid, item)
                        sighting_tracker.assign_global_id(tid, new_gid)
                        create_person_document(new_gid, sighting_tracker._frame_to_ts(frame_count), persons_col)
                    else:
                        best_gid, softmax_ratio, max_prob = gallery.classify_live_track(item['emb'], tau=TAU, exclude_gids=claimed_gids_this_frame)
                        if best_gid is not None and softmax_ratio >= TAU:
                            local_to_global_map[tid] = best_gid
                            claimed_gids_this_frame.add(best_gid)
                            gallery.add_observation(best_gid, item)
                            sighting_tracker.assign_global_id(tid, best_gid)
                        else:
                            udm.add(item)

        # Periodic DBSCAN
        if frame_count % 30 == 0 and len(udm.pool) >= UDM_MIN_SIZE:
            actions = udm.run_dbscan(gallery)
            for act, gid, cnt, track_ids in actions:
                if act == "MINT":
                    create_person_document(gid, sighting_tracker._frame_to_ts(frame_count), persons_col)
                for tid in track_ids:
                    sighting_tracker.assign_global_id(tid, gid)
                    local_to_global_map[tid] = gid
                if act == "FACE_VIP_BYPASS":
                    tqdm.write(f"🎫 VIP Pass: Face ID {gid:03d} recognized — pulling tracks from UDM! ({track_ids})")
                else:
                    tqdm.write(f"UDM: {act} → GID {gid:03d} (Tracks: {track_ids})")

        pbar.update(1)

# Close all remaining tracks
for tid in list(sighting_tracker.active_sightings.keys()):
    sighting_tracker.close_track(tid)

cap.release()
print(f"\n✅ Video Processing complete! Final Gallery: {gallery.num_classes()} distinct identities.")

## 6. CDN Upload & Cloud Sync

In [ ]:
print("\n☁️ Pushing Face/Body Modalities to CDN, Pinecone, & MongoDB...")

# ── Body sighting upload ──
def upload_sighting(sighting):
    """Upload body crop to ImageKit. Returns the updated sighting dict."""
    gid = sighting['global_id']
    tid = sighting['track_id']
    target_folder = f"/supermarket_reid/global_id_{gid:03d}/"

    img = sighting.get('best_crop_img')
    if img is not None:
        try:
            _, buffer = cv2.imencode('.jpg', img)
            filename = f"body_g{gid:03d}_t{tid}_{int(time.time())}.jpg"
            response = imagekit.files.upload(file=buffer.tobytes(), file_name=filename, folder=target_folder)
            sighting['best_crop_url'] = response.url if hasattr(response, 'url') else response.get('url', '')
        except Exception:
            sighting['best_crop_url'] = ""
        del sighting['best_crop_img']

    return sighting

# ── Face upload ──
def upload_face(face_data):
    """Upload face crop to ImageKit + upsert embedding to Pinecone. Returns dict for MongoDB."""
    gid = face_data['global_id']
    tid = face_data['track_id']
    target_folder = f"/supermarket_reid/global_id_{gid:03d}/"

    face_img = face_data.get('best_face_img')
    face_emb = face_data.get('best_face_emb')
    face_conf = face_data.get('best_face_conf', 0.0)

    result = {"global_id": gid, "track_id": tid, "camera_id": face_data.get("camera_id", CAMERA_ID)}

    if face_img is not None and face_img.size > 0:
        try:
            _, f_buffer = cv2.imencode('.jpg', face_img)
            f_filename = f"face_g{gid:03d}_t{tid}_{int(time.time())}.jpg"
            f_response = imagekit.files.upload(file=f_buffer.tobytes(), file_name=f_filename, folder=target_folder)
            face_url = f_response.url if hasattr(f_response, 'url') else f_response.get('url', '')

            if face_emb is not None:
                vector_id = f"face_g{gid:03d}_{int(time.time())}"
                face_index.upsert(
                    vectors=[{"id": vector_id, "values": face_emb.tolist(),
                              "metadata": {"global_id": gid, "face_url": face_url, "camera": face_data.get('camera_id', CAMERA_ID)}}]
                )
            result['url'] = face_url
            result['confidence'] = face_conf
        except Exception as e:
            print(f"Face Error: {e}")
            result['url'] = ""
            result['confidence'] = face_conf

    return result

# ── Upload body sightings (parallel) ──
completed_with_urls = []
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(upload_sighting, s)
               for s in sighting_tracker.completed_sightings if s.get('global_id') is not None]
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Body CDN Sync"):
        completed_with_urls.append(future.result())

# ── Upload faces (parallel) ──
completed_faces_with_urls = []
if IS_FACE_CAMERA and sighting_tracker.completed_faces:
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        futures = [executor.submit(upload_face, f)
                   for f in sighting_tracker.completed_faces if f.get('global_id') is not None]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Face CDN Sync"):
            completed_faces_with_urls.append(future.result())

# ── MongoDB: push sightings ──
print("Updating MongoDB (sightings)...")
for s in completed_with_urls:
    gid = s.pop('global_id')
    persons_col.update_one({"global_id": gid}, {
        "$push": {"sightings": s},
        "$set": {"last_seen_ts": float(s["exit_ts"])}
    })

# ── MongoDB: push faces ──
if completed_faces_with_urls:
    print("Updating MongoDB (faces)...")
    for f in completed_faces_with_urls:
        gid = f.pop('global_id')
        f.pop('track_id', None)
        timestamp = f.pop('timestamp', time.time())
        persons_col.update_one({"global_id": gid}, {
            "$push": {"faces": {
                "url": f.get('url', ''),
                "confidence": f.get('confidence', 0.0),
                "timestamp": float(timestamp),
                "camera_id": f.get('camera_id', CAMERA_ID)
            }}
        })

# Serialize gallery for Module 3
print("Serializing Output Buffer for Module 3...")
serializable = {}
for gid in gallery.gids:
    data = gallery.classes[gid]
    serializable[gid] = {
        "embeddings": [e.cpu().numpy() for e in data["embs"]],
        "ratios": data["ratios"],
        "det_scores": data["scores"],
        "filepaths": [str(p) for p in data["filepaths"]],
    }

with open(SAVE_BUFFER_PATH, 'wb') as f:
    pickle.dump(serializable, f)

shutil.make_archive('/kaggle/working/saved_identities', 'zip', str(DB_IMAGES_DIR))
print("✅ Done! Face and Body pipelines saved separately.")
